# 📘 **NOAI 2026 - Programming Task 2**
* **Subject:** Deep Learning (Computer Vision)
* **Title:** The Segmentation Fracture (Debugging U-Net)
* **Total:** 20 Marks
  * **Part 1: The Architecture (10 Marks)** - Fixing the Padding Mismatch
  * **Part 2: The Metric (10 Marks)** - Re-implementing Mean IoU
  * **Part 3: System Verification (No Marks Allocated)** - Final Integrity Check

## **1. The Scenario**
You are the Lead Engineer at *OrbitalVision*, a startup processing high-resolution satellite imagery to track deforestation. Your team uses a standard **U-Net** architecture to segment images into two classes: **Urban (0)** and **Forest (1)**.

A junior intern recently tried to optimize the model for speed by removing "unnecessary" padding. However, this broke the geometry of the network. Now, the training pipeline crashes with `RuntimeError: Sizes of tensors must match` because the feature maps in the Decoder are larger than the ones from the Encoder.

**Your Mission:**
1.  **Fix the Architecture:** Restore the padding logic so the skip connections align perfectly.
2.  **Restore the Metric:** The intern also deleted the evaluation code. You must re-implement the **Mean Intersection over Union (mIoU)** from scratch (no Scikit-Learn allowed).

### **📚 Supplementary Reading**

* **[A Guide to Convolution Arithmetic](https://arxiv.org/abs/1603.07285)** (Dumoulin & Visin)
* **[U-Net: Convolutional Networks for Biomedical Image Segmentation](https://arxiv.org/abs/1505.04597)** (Ronneberger et al.)
* **[Guide to Intersection over Union (IoU)](https://www.v7labs.com/blog/intersection-over-union-guide)**

## **2. Setup & Rules**
- **Allowed Libraries:** `torch`, `torch.nn`, `torch.nn.functional`, `numpy`.
- **Banned Libraries:** `sklearn`, `torchmetrics`, or any other high-level metric wrappers.
- **Input Data:** We provide a mock random tensor `(B, 3, 256, 256)` to test your forward pass.

In [ ]:
# --- CELL 1: SETUP (DO NOT EDIT) ---
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# Set seed for reproducibility
torch.manual_seed(2026)

print("Setup Complete. Device:", "cuda" if torch.cuda.is_available() else "cpu")

## **Part 1: The Architecture (10 Marks)**

### **Question 2.1: The "Valid Padding" Fracture**
The intern used `padding=0` ("Valid" convolution) to save compute. This caused a critical geometric issue: **The Feature Maps Shrink.**

* **Input:** $256 \times 256$
* **Encoder Layer 1:** $254 \times 254$ (Lost 2 pixels border)
* **...**
* **Bottleneck Output:** The feature map is significantly smaller than the input.

When the Decoder upsamples the image, it reaches a size (e.g., $114 \times 114$) that is smaller than the corresponding Skip Connection from the Encoder (e.g., $122 \times 122$). Pytorch cannot concatenate tensors of different sizes.

**Your Task:**
1.  **Analyze** the shape mismatch in the `forward` method.
2.  **Implement** the `center_crop(layer, target_shape)` method. It must slice the larger Encoder feature map (`layer`) from the center so it matches the height and width of the Decoder feature map (`target_shape`).
3.  **Verify** that the channel dimensions align for concatenation.

In [ ]:
### QUESTION 2.1 START: Fix the Architecture (15 pts)

class BrokenUNet(nn.Module):
    def __init__(self, in_channels=3, out_classes=2):
        super(BrokenUNet, self).__init__()
        
        # --- ENCODER (Intern removed padding to 'save compute') ---
        # Note: kernel=3, padding=0 causes spatial dimension reduction by 2 pixels per layer.
        self.enc1_1 = nn.Conv2d(in_channels, 64, kernel_size=3, padding=0) 
        self.enc1_2 = nn.Conv2d(64, 64, kernel_size=3, padding=0)
        self.pool1 = nn.MaxPool2d(2) # Downsample by 2
        
        self.enc2_1 = nn.Conv2d(64, 128, kernel_size=3, padding=0)
        self.enc2_2 = nn.Conv2d(128, 128, kernel_size=3, padding=0)
        self.pool2 = nn.MaxPool2d(2) # Downsample by 2
        
        # --- BOTTLENECK ---
        self.bot1 = nn.Conv2d(128, 256, kernel_size=3, padding=0)
        self.bot2 = nn.Conv2d(256, 256, kernel_size=3, padding=0)
        
        # --- DECODER ---
        self.upconv1 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        
        # Decoder Convolutions
        # Expects 256 input channels (128 from upconv + 128 from skip)
        self.dec1_1 = nn.Conv2d(256, 128, kernel_size=3, padding=0)
        self.dec1_2 = nn.Conv2d(128, 64, kernel_size=3, padding=0)
        
        self.final = nn.Conv2d(64, out_classes, kernel_size=1)

    def center_crop(self, layer, target_shape):
        """
        Crop the layer to match target dimensions (center-aligned).
        
        Why needed: With padding=0, encoder feature maps are LARGER than 
        decoder feature maps after upsampling. We need to crop the encoder 
        features from the center to match decoder size for concatenation.
        
        Args:
            layer: Tensor of shape (B, C, H_large, W_large) - the skip connection from encoder
            target_shape: torch.Size (B, C, H_small, W_small) - the upsampled decoder tensor
        
        Returns:
            Cropped tensor of shape (B, C, H_small, W_small)
        
        Example:
            layer shape: (1, 128, 122, 122)  <- from encoder
            target_shape: (1, 128, 114, 114) <- from decoder after upsampling
            Need to crop: from [122x122] to [114x114]
            Center crop: start at (122-114)//2 = 4, end at 4+114 = 118
            Return: layer[:, :, 4:118, 4:118]
        """
        _, _, h_layer, w_layer = layer.shape
        target_h = target_shape[2]
        target_w = target_shape[3]
        offset_h = (h_layer - target_h) // 2
        offset_w = (w_layer - target_w) // 2
        return layer[:, :, offset_h:offset_h + target_h, offset_w:offset_w + target_w]

    def forward(self, x):
        # 1. Encoder 1
        x1 = F.relu(self.enc1_1(x))
        x1 = F.relu(self.enc1_2(x1))
        x1_p = self.pool1(x1)
        
        # 2. Encoder 2
        x2 = F.relu(self.enc2_1(x1_p))
        x2 = F.relu(self.enc2_2(x2))
        x2_p = self.pool2(x2)
        
        # 3. Bottleneck
        bot = F.relu(self.bot1(x2_p))
        bot = F.relu(self.bot2(bot))
        
        # 4. Decoder 1
        d1 = self.upconv1(bot)
        
        # --- CRASH POINT: Shape Mismatch ---
        # x2 is larger (from Encoder). d1 is smaller (from Bottleneck).
        # You must use self.center_crop() here to make x2 match d1.
        
        # TODO: UNCOMMENT the line below after implementing center_crop()
        # x2_cropped = self.center_crop(x2, d1.shape)
        x2_cropped = self.center_crop(x2, d1.shape)  # This placeholder will cause a crash - you must fix it!
        
        # Concatenate along channel axis (dim=1)
        cat1 = torch.cat((x2_cropped, d1), dim=1)
        
        dec1 = F.relu(self.dec1_1(cat1))
        dec1 = F.relu(self.dec1_2(dec1))
        
        return self.final(dec1)

### QUESTION 2.1 END

In [ ]:
# --- SELF-CHECK: RUN THIS TO TEST YOUR FIX ---
# If your center_crop is correct, this cell will print a Success message.

model = BrokenUNet()
dummy_input = torch.randn(1, 3, 256, 256)

try:
    output = model(dummy_input)
    print("✅ Forward Pass Successful!")
    print(f"Input Shape:  {dummy_input.shape}")
    print(f"Output Shape: {output.shape}")
    print("-" * 30)
    print("Note: The output is smaller than the input (e.g. 196x196) because padding=0.")
    print("This is expected behavior for this specific architecture.")
except Exception as e:
    print(f"❌ CRASH REPORT: {e}")

### **Question 2.1b: Why Center Cropping? (Conceptual - No Coding)**

After fixing the code, answer this question in your own words:

**Q:** The intern removed padding to "save compute". This caused feature maps to shrink. 
You fixed it with center cropping. But what are the **trade-offs** of this approach?

**Think about:**
- What information do we lose when we crop?
- Could we use a different solution (e.g., zero-padding, interpolation)?
- Why does the original U-Net paper use padding=1 instead?

**Your Answer (2-3 sentences):**

*[Center cropping discards border pixels from the encoder feature maps, losing spatial context at the edges of the image. This means the output segmentation map is smaller than the input (e.g. ~196×196 from a 256×256 input), so edge regions receive no prediction. The original U-Net paper uses padding=1 to preserve spatial dimensions throughout, avoiding this information loss entirely — alternatively, one could use interpolation (F.interpolate) to resize the decoder features upward instead of cropping the encoder features downward, though this introduces artificial values rather than discarding real ones.]*

---

**Hint:** There's no single "correct" answer - this tests your understanding of design choices.

## **Part 2: The Metric (10 Marks)**

### **Question 2.2: The "IoU" Standard**

**Why IoU instead of Accuracy?**
If 90% of a satellite map is Forest, a dumb model that always predicts "Forest" gets 90% accuracy but has zero utility. IoU measures actual overlap between prediction and ground truth.

---

### **The Formula**

For a single class:

$\displaystyle \text{IoU}_{\text{class}} = \frac{\text{Intersection}}{\text{Union}} = \frac{TP}{TP + FP + FN}$

Where:
- **Intersection** = Pixels where BOTH prediction AND target are this class (Logical AND)
- **Union** = Pixels where EITHER prediction OR target is this class (Logical OR)

**Mean IoU (mIoU):** Average IoU across all classes (skip classes where union = 0)

---

### **PyTorch Implementation Pattern**
```python
# For class c:
pred_mask = (pred == c)    # Boolean tensor: True where predicted as class c
true_mask = (target == c)  # Boolean tensor: True where actually class c

intersection = (pred_mask & true_mask).sum()  # Logical AND, then count
union = (pred_mask | true_mask).sum()         # Logical OR, then count

iou = intersection / union  # ⚠️ Will be NaN if union == 0!
```

---

### **Your Task**

Implement `calculate_miou` with these constraints:
1. **Vectorized Operations Only:** You may loop over classes (0, 1), but **NO** loops over pixels or batches
2. **Handle Missing Classes:** If union == 0 (class doesn't exist in target), skip it to avoid `NaN`

In [ ]:
### QUESTION 2.2 START: Implement Mean IoU (15 pts)

def calculate_miou(pred_mask, true_mask, num_classes=2):
    """
    Computes Mean Intersection over Union.
    Args:
        pred_mask: (B, H, W) Integer Tensor - predicted class labels
        true_mask: (B, H, W) Integer Tensor - ground truth class labels
        num_classes: Number of classes (e.g., 2 for Urban/Forest)
    
    Returns:
        float: Mean IoU across all classes
    
    Algorithm:
        For each class c in [0, num_classes-1]:
            1. Create boolean masks: (pred == c) and (true == c)
            2. Intersection = count where BOTH are True  [use & operator]
            3. Union = count where AT LEAST ONE is True  [use | operator]
            4. IoU = Intersection / Union
            5. If Union == 0 (class doesn't exist), skip this class [use continue]
        Return: Average of all valid IoUs [sum(ious) / len(ious)]
    
    Vectorization requirement:
        - NO loops over pixels or batches
        - Flatten tensors first: pred_flat = pred_mask.view(-1)
        - Use boolean indexing: (pred_flat == cls) creates boolean tensor
        - Use .sum() to count True values
    """
    # TODO: Flatten tensors to (N,) using .view(-1) or .flatten()
    pred_flat = pred_mask.view(-1)
    true_flat = true_mask.view(-1)
    
    ious = []
    
    # TODO: Loop over classes
    for cls in range(num_classes):
        pred_mask_c = (pred_flat == cls)
        true_mask_c = (true_flat == cls)

        intersection = (pred_mask_c & true_mask_c).sum().item()
        union = (pred_mask_c | true_mask_c).sum().item()

        if union == 0:
            continue

        ious.append(intersection / union)

    if len(ious) == 0:
        return 0.0

    return sum(ious) / len(ious)

### QUESTION 2.2 END

In [ ]:
# --- SELF-CHECK: RUN THIS TO TEST YOUR METRIC ---
# We create a fake perfect prediction and a bad prediction
# 2x2 Image
fake_pred = torch.tensor([[[0, 0], [1, 1]]]) 
# One pixel differs
fake_true = torch.tensor([[[0, 1], [1, 1]]]) 

# Class 0 (Urban): Pred=[T, T, F, F], True=[T, F, F, F] -> IoU = 1/2 = 0.5
# Class 1 (Forest): Pred=[F, F, T, T], True=[F, T, T, T] -> IoU = 2/3 = 0.66
# Mean IoU = (0.5 + 0.66) / 2 = 0.5833

try:
    score = calculate_miou(fake_pred, fake_true, num_classes=2)
    print(f"Calculated mIoU: {score:.4f}")
    print(f"Expected mIoU:   0.5833")
    
    if 0.58 < score < 0.59:
        print("✅ Metric Check Passed!")
    else:
        print("❌ Metric Check Failed: Value mismatch.")
        
except Exception as e:
    print(f"❌ Metric Check Crashed: {e}")

In [ ]:
# --- ADDITIONAL TEST: Missing Class Handling ---
# This tests the Union == 0 edge case

# Create scenario where class 1 (Forest) doesn't exist in target
pred_missing = torch.tensor([[[0, 0], [0, 1]]])   # Some predictions of class 1
true_missing = torch.tensor([[[0, 0], [0, 0]]])   # But no class 1 in ground truth

# Class 0: Both pred and true have it → IoU should be computed
# Class 1: Union = 0 (doesn't exist in target) → Should be skipped

try:
    score_missing = calculate_miou(pred_missing, true_missing, num_classes=2)
    print(f"mIoU with missing class: {score_missing:.4f}")
    print("Expected: Only class 0 IoU counted (should be 0.75)")
    
    # Class 0: pred=[T,T,T,F], true=[T,T,T,T] → Intersection=3, Union=4, IoU=0.75
    if 0.74 < score_missing < 0.76:
        print("✅ Missing class handling correct!")
    else:
        print("❌ Missing class handling failed")
        
except Exception as e:
    print(f"❌ Test crashed: {e}")

## **Part 3: System Verification (No Marks Allocated)**

### **Final Integrity Check**
Before you submit, run the cell below to verify that your fixed U-Net (`BrokenUNet`) and your metric (`calculate_miou`) are compatible.

**Why is this important?**
Even if your logic seems correct, a shape mismatch during the forward pass will cause the auto-grader to fail your submission entirely. This is your "Pre-Flight Check."

**Success Criteria:**
1.  The model accepts a $256 \times 256$ input without crashing.
2.  The output tensor has the correct shape (likely smaller than input due to valid padding, e.g., $196 \times 196$, but this is expected).

In [ ]:
# --- FINAL SYSTEM CHECK ---
# Run this cell to ensure your model is ready for submission.

print("Initiating System Integrity Check...")

try:
    # 1. Instantiate the model
    model = BrokenUNet(in_channels=3, out_classes=2)
    
    # 2. Create dummy input
    x = torch.randn(1, 3, 256, 256)
    
    # 3. Run Forward Pass
    y = model(x)
    
    print("✅ System Online: Forward pass successful.")
    print(f"   Input Shape:  {x.shape}")
    print(f"   Output Shape: {y.shape}")
    
    if y.shape[2] < x.shape[2]:
        print("\n   (Note: Output is smaller than input due to valid padding. This is normal.)")

except Exception as e:
    print(f"❌ System Failure: {e}")
    print("   CRITICAL: Your model is still broken. Go back to Question 2.1.")

## End of NOAI 2026 - Programming Question 2
---